# Оптимальное управление угловым движением спутника


Вариант 35: $J = 2500$ кг·м$^2$, $M_m = 45$ Н·м, $\nu_0 = -40^\circ$, $\dot\nu_0 = 25^\circ/с$.

## 1. Исходные данные

Уравнения движения спутника:

$$
\dot\nu = \omega, \qquad \dot\omega = k\mu,
$$

где $\omega = \dot\nu$, $\mu \in \{-1, +1\}$, $k = M_m / J$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.grid"] = True

In [ ]:
# Исходные данные
J = 2500          # кг*м^2
M_m = 45         # Н*м
k = M_m / J      # рад/с^2

deg = np.pi / 180
nu0 = -40 * deg       # рад
nu_dot0 = 25 * deg    # рад/с

EPS = 1e-6

print(f"k = {k:.6f} рад/с^2")
print(f"nu0 = {nu0:.6f} рад")
print(f"nu_dot0 = {nu_dot0:.6f} рад/с")

## 2. Линия переключения и закон управления

Линия переключения для оптимального управления:

$$
\omega =
\begin{cases}
-\sqrt{2k\nu}, & \nu \ge 0,\\
\sqrt{-2k\nu}, & \nu < 0.
\end{cases}
$$

Закон управления выбирается по положению текущей точки относительно линии переключения.

In [ ]:
def switch_line(nu):
    """Линия переключения omega = f(nu). Работает со скалярами и массивами."""
    nu = np.asarray(nu, dtype=float)
    return np.where(
        nu >= 0,
        -np.sqrt(np.maximum(0.0, 2 * k * nu)),
        np.sqrt(np.maximum(0.0, -2 * k * nu)),
    )


def switch_line_scalar(nu):
    return float(switch_line(np.array([nu]))[0])


def control(nu, nu_dot):
    """Оптимальное управление mu = +/-1."""
    diff = nu_dot - switch_line_scalar(nu)
    if abs(diff) <= EPS:
        return 1 if nu >= 0 else -1
    return -1 if diff > 0 else 1

## 3. Аналитический расчет $\Delta t_1$ и $\Delta t_2$

Для первой дуги при $\mu = -1$ используется интеграл движения:

$$
\omega^2 + 2k\nu = C_0.
$$

Точка переключения определяется пересечением первой дуги с линией переключения:

$$
\nu_1 = \frac{C_0}{4k}, \qquad
\omega_1 = -\sqrt{2k\nu_1}.
$$

Времена участков:

$$
\Delta t_1 = \frac{\omega_1 - \omega_0}{-k}, \qquad
\Delta t_2 = \frac{0 - \omega_1}{k}.
$$

In [ ]:
C0 = nu_dot0**2 + 2 * k * nu0
nu1 = C0 / (4 * k)
nu_dot1 = -np.sqrt(2 * k * nu1)

dt1 = (nu_dot1 - nu_dot0) / (-k)
dt2 = (0 - nu_dot1) / k
T_opt = dt1 + dt2

print(f"C0 = {C0:.9f}")
print(f"nu1 = {nu1:.9f} рад")
print(f"nu_dot1 = {nu_dot1:.9f} рад/с")
print(f"Delta t1 = {dt1:.3f} с")
print(f"Delta t2 = {dt2:.3f} с")
print(f"Topt = {T_opt:.3f} с")

## 4. Численное моделирование

Система интегрируется методом Рунге--Кутты 4-го порядка. Управление на каждом шаге пересчитывается по текущему положению точки на фазовой плоскости.

In [ ]:
dt = 0.0005
t = 0.0
nu = nu0
nu_dot = nu_dot0

t_arr = []
nu_arr = []
nu_dot_arr = []
u_arr = []

switch_time = None
prev_u = control(nu, nu_dot)
max_iter = 10_000_000

for _ in range(max_iter):
    u_now = control(nu, nu_dot)
    if switch_time is None and u_now != prev_u:
        switch_time = t
    prev_u = u_now

    t_arr.append(t)
    nu_arr.append(nu)
    nu_dot_arr.append(nu_dot)
    u_arr.append(u_now)

    h = dt

    k1_nu = h * nu_dot
    k1_nu_dot = h * k * u_now

    k2_nu = h * (nu_dot + k1_nu_dot / 2)
    k2_nu_dot = h * k * u_now

    k3_nu = h * (nu_dot + k2_nu_dot / 2)
    k3_nu_dot = h * k * u_now

    k4_nu = h * (nu_dot + k3_nu_dot)
    k4_nu_dot = h * k * u_now

    nu += (k1_nu + 2 * k2_nu + 2 * k3_nu + k4_nu) / 6
    nu_dot += (k1_nu_dot + 2 * k2_nu_dot + 2 * k3_nu_dot + k4_nu_dot) / 6
    t += h

    if abs(nu_dot - switch_line_scalar(nu)) <= EPS:
        nu_dot = switch_line_scalar(nu)

    if abs(nu) < 1e-6 and abs(nu_dot) < 1e-6:
        break
else:
    raise RuntimeError("Не достигнут критерий остановки")

t_arr = np.array(t_arr)
nu_arr = np.array(nu_arr)
nu_dot_arr = np.array(nu_dot_arr)
u_arr = np.array(u_arr)

dt1_sim = switch_time
dt2_sim = t - switch_time if switch_time is not None else np.nan
T_sim = t

print(f"Delta t1 по графику/модели = {dt1_sim:.3f} с")
print(f"Delta t2 по графику/модели = {dt2_sim:.3f} с")
print(f"Tsim = {T_sim:.3f} с")

## 5. Фазовые диаграммы при постоянном управлении

Для постоянных управлений $\mu = +1$ и $\mu = -1$ фазовые кривые строятся по интегралам движения:

$$
\frac{x_2^2}{2} = k\mu x_1 + C.
$$

In [ ]:
x_min = min(-4.0, nu0, nu1) - 0.5
x_max = max(4.0, nu0, nu1) + 0.5
x_grid = np.linspace(x_min, x_max, 2000)

C_plus = nu_dot0**2 / 2 - k * nu0
C_minus = nu_dot0**2 / 2 + k * nu0

rad_plus = 2 * k * x_grid + 2 * C_plus
rad_minus = -2 * k * x_grid + 2 * C_minus

y_plus = np.where(rad_plus >= 0, np.sqrt(np.maximum(0.0, rad_plus)), np.nan)
y_minus = np.where(rad_minus >= 0, np.sqrt(np.maximum(0.0, rad_minus)), np.nan)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_grid, y_plus, color="royalblue", linewidth=1.8, label=r"$\mu=+1$")
ax.plot(x_grid, -y_plus, color="royalblue", linewidth=1.8)
ax.plot(x_grid, y_minus, color="crimson", linewidth=1.8, label=r"$\mu=-1$")
ax.plot(x_grid, -y_minus, color="crimson", linewidth=1.8)
ax.scatter([nu0], [nu_dot0], color="black", zorder=4, label="начальная точка")
ax.axhline(0, color="black", linewidth=0.7)
ax.axvline(0, color="black", linewidth=0.7)
ax.set_title(r"Фазовые диаграммы при $\mu=\pm1$")
ax.set_xlabel(r"$\nu$, рад")
ax.set_ylabel(r"$\dot\nu$, рад/с")
ax.set_xlim(x_min, x_max)
ax.legend(loc="best")
plt.show()

## 6. Графики

In [ ]:
x_min = min(-4.0, float(nu_arr.min()), nu0, nu1) - 0.5
x_max = max(4.0, float(nu_arr.max()), nu0, nu1) + 0.5
x_grid = np.linspace(x_min, x_max, 2000)
phase_parabola = np.sqrt(np.maximum(0.0, 2 * k * np.abs(x_grid)))

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
ax_phase, ax_v, ax_vdot, ax_vvs = axes.ravel()

# 1. Фазовая плоскость
ax_phase.plot(x_grid, phase_parabola, color="0.55", linestyle=":", linewidth=1.4)
ax_phase.plot(x_grid, -phase_parabola, color="0.55", linestyle=":", linewidth=1.4)
ax_phase.plot(x_grid, switch_line(x_grid), color="red", linewidth=1.8, label="линия переключения")
ax_phase.plot(nu_arr, nu_dot_arr, color="blue", linewidth=2.4, label="оптимальная")
ax_phase.axhline(0, color="black", linewidth=0.7)
ax_phase.axvline(0, color="black", linewidth=0.7)
ax_phase.set_title("Фазовая плоскость")
ax_phase.set_xlabel(r"$\nu$ (рад)")
ax_phase.set_ylabel(r"$\dot\nu$ (рад/с)")
ax_phase.set_xlim(x_min, x_max)
ax_phase.legend(loc="best")

# 2. nu(t)
ax_v.plot(t_arr, nu_arr, color="blue", linewidth=2)
ax_v.set_title(r"$\nu(t)$")
ax_v.set_xlabel("t, с")
ax_v.set_ylabel(r"$\nu$, рад")

# 3. nu_dot(t)
ax_vdot.plot(t_arr, nu_dot_arr, color="green", linewidth=2)
ax_vdot.set_title(r"$\dot\nu(t)$")
ax_vdot.set_xlabel("t, с")
ax_vdot.set_ylabel(r"$\dot\nu$, рад/с")

# 4. nu_dot(nu) с линией переключения
ax_vvs.plot(nu_arr, nu_dot_arr, color="blue", linewidth=2.4, label="оптимальная")
ax_vvs.plot(x_grid, switch_line(x_grid), color="red", linewidth=1.8, label="линия переключения")
ax_vvs.axhline(0, color="black", linewidth=0.7)
ax_vvs.axvline(0, color="black", linewidth=0.7)
ax_vvs.set_title(r"$\dot\nu(\nu)$ с линией переключения")
ax_vvs.set_xlabel(r"$\nu$, рад")
ax_vvs.set_ylabel(r"$\dot\nu$, рад/с")
ax_vvs.set_xlim(x_min, x_max)
ax_vvs.legend(loc="best")

plt.tight_layout()
plt.show()

## 7. Сравнение результатов

In [ ]:
rows = [
    ("Delta t1", dt1, dt1_sim, abs(dt1 - dt1_sim)),
    ("Delta t2", dt2, dt2_sim, abs(dt2 - dt2_sim)),
    ("T", T_opt, T_sim, abs(T_opt - T_sim)),
]

print(f"{'Параметр':<12} {'Расчет':>12} {'Модель':>12} {'Разность':>12}")
print("-" * 52)
for name, calc, model, diff in rows:
    print(f"{name:<12} {calc:>12.3f} {model:>12.3f} {diff:>12.3f}")